# Script 5: Platform Host Functional Goal vs General Public Ethical Goal

### The Conflict
This notebook encodes a conflict between two different stakeholders.

From Chapter 2:
- **Platform host functional goal:** distribute as many models and applications as widely as possible
- **General public ethical goal:** not be exposed to harmful content hrough publicly accessible platforms

These conflict when a platform hosts a derivative model that has had its safety mechanisms removed, without any screening process.

This is distinct from Notebooks 1 and 4. Notebook 1 tested the system design conflict. Notebook 4 tested the developer's internal conflict. 
This notebook tests the distribution layer — the point at which Hugging Face decides what to host.

### Domain Elements
| Variable | Type | Meaning |
|---|---|---|
| IsDerivative | Bool | Is this model built on Stable Diffusion |
| SafetyRemoved | Bool | Has the safety checker been removed |
| Screened | Bool | Did the platform review this model before hosting |
| PubliclyHosted | Bool/Int | Is the model publicly accessible on the platform |

### The Formal Constraint
It must never be the case that a model is publicly hosted when it is a derivative of Stable Diffusion AND its safety has been removed AND 
it has not been screened.

Formally: NOT (PubliclyHosted AND IsDerivative AND SafetyRemoved AND NOT Screened)

### The Ethical School
This conflict maps to utilitarianism. The platform's value of wide distribution benefits many users. The harm caused by hosting unscreened models with removed safety falls on specific identifiable individuals — women and minors — who receive none of the benefit. A utilitarian calculation that accounts for the concentration of harm cannot justify unscreened hosting.

### Imports

In [1]:
from z3 import *
import requests

### Part 1: Encoding the Conflict

We force PubliclyHosted=True for every model to represent the platform's functional goal of maximum distribution. We then add the ethical constraint on top.

In [ ]:
def check_platform_conflict(scenario_name, models):
    N = len(models)

    IsDerivative  = [Bool(f'IsDerivative_{i}')  for i in range(N)]
    SafetyRemoved = [Bool(f'SafetyRemoved_{i}') for i in range(N)]
    Screened      = [Bool(f'Screened_{i}')      for i in range(N)]
    Hosted        = [Bool(f'Hosted_{i}')        for i in range(N)]

    s = Solver()

    for i, m in enumerate(models):
        s.add(IsDerivative[i]  == m['derivative'])
        s.add(SafetyRemoved[i] == m['safety_removed'])
        s.add(Screened[i]      == m['screened'])

    # FUNCTIONAL GOAL: host every model
    for i in range(N):
        s.add(Hosted[i] == True)

    # ETHICAL CONSTRAINT: never host an unscreened derivative 
    for i in range(N):
        s.add(Not(And(
            Hosted[i],
            IsDerivative[i],
            SafetyRemoved[i],
            Not(Screened[i])
        )))

    result = s.check()

    print(f"\n{'='*65}")
    print(f"Scenario: {scenario_name}")
    print(f"{'='*65}")
    print(f"  {'t':<5} {'Derivative':<14} {'Safety Off':<14} {'Screened':<12} {'Host?'}")
    print(f"  {'-'*60}")
    for i, m in enumerate(models):
        print(f"  t={i}   {str(m['derivative']):<14} {str(m['safety_removed']):<14} "
              f"{str(m['screened']):<12} True (functional goal)")

    print(f"\n  Z3 Result: {result}")
    if result == sat:
        print("  NO CONFLICT: Platform functional goal and public ethical goal compatible.")
    else:
        print("  CONFLICT DETECTED: Platform cannot host all models")
        print("  without violating the general public's ethical goal.")
        print("  The functional goal demands hosting.")
        print("  The ethical goal demands screening before hosting.")
        print("  Violated principle: Utilitarian — concentrated harm on")
        print("  identifiable individuals outweighs broad distribution benefit.")

### Scenario 1: All models are screened before hosting

Every derivative model with safety removed has been reviewed. Platform functional goal and public ethical goal are compatible.

In [ ]:
scenario_1 = [
    {'derivative': True,  'safety_removed': True,  'screened': True},
    {'derivative': True,  'safety_removed': False, 'screened': True},
    {'derivative': False, 'safety_removed': False, 'screened': True},
    {'derivative': True,  'safety_removed': True,  'screened': True},
]
check_platform_conflict("All models screened — no conflict expected", scenario_1)

### Scenario 2: Unscreened derivative models with safety removed

This reflects the current state of the Hugging Face ecosystem as documented in Notebook 3. Derivative models with safety mechanisms removed are publicly hosted without any documented screening process.

Z3 returns UNSAT. The platform cannot simultaneously satisfy its functional goal of hosting everything and the general public's 
ethical goal of protection from unscreened harmful content.

In [3]:
scenario_2 = [
    {'derivative': True,  'safety_removed': False, 'screened': True},
    {'derivative': True,  'safety_removed': True,  'screened': False},
    {'derivative': True,  'safety_removed': True,  'screened': False},
    {'derivative': False, 'safety_removed': False, 'screened': True},
]
check_platform_conflict("Unscreened models with safety removed — conflict expected", scenario_2)


Scenario: Unscreened models with safety removed — conflict expected
  t     Derivative     Safety Off     Screened     Host?
  ------------------------------------------------------------
  t=0   True           False          True         True (functional goal)
  t=1   True           True           False        True (functional goal)
  t=2   True           True           False        True (functional goal)
  t=3   False          False          True         True (functional goal)

  Z3 Result: unsat
  CONFLICT DETECTED: Platform cannot host all models
  without violating the general public's ethical goal.
  The functional goal demands hosting.
  The ethical goal demands screening before hosting.
  Violated principle: Utilitarian — concentrated harm on
  identifiable individuals outweighs broad distribution benefit.


### Part 2: Grounding in Real Data

Scenario 2 above is not hypothetical. Notebook 3 found derivative models built on Stable Diffusion with NSFW content publicly hosted on Hugging Face. We now construct the scenarios from that real data and run the same conflict check.

The models found in Notebook 3 are treated as facts. The screening status is derived from Hugging Face's own documentation — no formal screening process for derivative model safety status is documented.

In [4]:
# Real data from Notebook 3 audit

real_models = [
    {'name': 'Heartsync/NSFW-Uncensored-photo', 
     'derivative': True, 'safety_removed': True, 'screened': False},
    {'name': 'BoobyBoobs/Flux_Lustly_AI_Uncensored_NSFW_V1', 
     'derivative': True, 'safety_removed': True, 'screened': False},
    {'name': 'Landol4/nsfw-face-swap', 
     'derivative': True, 'safety_removed': True, 'screened': False},
    {'name': 'Menyu/wainsfw', 
     'derivative': True, 'safety_removed': True, 'screened': False},
    {'name': 'Heartsync/Adult', 
     'derivative': True, 'safety_removed': True, 'screened': False},
]

N = len(real_models)

IsDerivative  = [Bool(f'IsDerivative_r{i}')  for i in range(N)]
SafetyRemoved = [Bool(f'SafetyRemoved_r{i}') for i in range(N)]
Screened      = [Bool(f'Screened_r{i}')      for i in range(N)]
Hosted        = [Bool(f'Hosted_r{i}')        for i in range(N)]

s = Solver()

for i, m in enumerate(real_models):
    s.add(IsDerivative[i]  == m['derivative'])
    s.add(SafetyRemoved[i] == m['safety_removed'])
    s.add(Screened[i]      == m['screened'])
    s.add(Hosted[i]        == True)

for i in range(N):
    s.add(Not(And(
        Hosted[i],
        IsDerivative[i],
        SafetyRemoved[i],
        Not(Screened[i])
    )))

result = s.check()

print("Real Data Conflict Check")
print("="*60)
print("Models: Derivative SD applications with safety removed,")
print("        no documented screening, publicly hosted.")
print()
for m in real_models:
    print(f"  {m['name']}")
print()
print(f"Z3 Result: {result}")
if result == unsat:
    print("UNSAT: The platform cannot host these models without")
    print("violating the general public's ethical goal.")
    print("These are not hypothetical cases. They are currently")
    print("publicly accessible on Hugging Face.")

Real Data Conflict Check
Models: Derivative SD applications with safety removed,
        no documented screening, publicly hosted.

  Heartsync/NSFW-Uncensored-photo
  BoobyBoobs/Flux_Lustly_AI_Uncensored_NSFW_V1
  Landol4/nsfw-face-swap
  Menyu/wainsfw
  Heartsync/Adult

Z3 Result: unsat
UNSAT: The platform cannot host these models without
violating the general public's ethical goal.
These are not hypothetical cases. They are currently
publicly accessible on Hugging Face.


### Part 3: Conflict Resolution

Z3 confirmed the conflict. Now we resolve it by letting Z3 decide which models to host subject to the ethical constraint as a hard rule. 
Z3 maximises hosting while blocking unscreened derivative models with safety removed.

In [5]:
def resolve_platform_conflict(models):
    N = len(models)

    IsDerivative = [Bool(f'IsDerivative_res{i}') for i in range(N)]
    SafetyRemoved = [Bool(f'SafetyRemoved_res{i}') for i in range(N)]
    Screened = [Bool(f'Screened_res{i}') for i in range(N)]
    Hosted = [Int(f'Hosted_res{i}') for i in range(N)]

    opt = Optimize()

    for i, m in enumerate(models):
        opt.add(IsDerivative[i] == m['derivative'])
        opt.add(SafetyRemoved[i] == m['safety_removed'])
        opt.add(Screened[i] == m['screened'])
        opt.add(Or(Hosted[i] == 0, Hosted[i] == 1))

    # ETHICAL CONSTRAINT: hard rule
    for i in range(N):
        opt.add(
            Implies(
                And(IsDerivative[i], SafetyRemoved[i], Not(Screened[i])),
                Hosted[i] == 0
            )
        )

    # FUNCTIONAL GOAL: soft objective
    opt.maximize(Sum(Hosted))

    result = opt.check()
    if result != sat:
        print(f"Z3 could not find a solution: {result}")
        return

    model_sol = opt.model()

    print("Conflict Resolution: Platform Hosting with Ethical Constraint")
    print("="*65)
    print(f"  {'Model':<45} {'Hosted':<10} Reason")
    print(f"  {'-'*75}")

    hosted_count = 0
    blocked_count = 0

    for i, m in enumerate(models):
        h = model_sol[Hosted[i]].as_long()
        if h == 1:
            reason = "Functional goal met"
            hosted_count += 1
        else:
            reason = "Ethical constraint overrides"
            blocked_count += 1
        print(f"  {m['name']:<45} {str(h):<10} {reason}")

    print(f"\n  Hosted:  {hosted_count}/{N}")
    print(f"  Blocked: {blocked_count}/{N}")
    if blocked_count > 0:
        print(f"\n  Unscreened derivative models with safety removed are blocked.")
        print(f"  The platform's functional goal is partially met.")
        print(f"  The general public's ethical goal takes precedence.")

In [6]:

resolve_platform_conflict(real_models)

Conflict Resolution: Platform Hosting with Ethical Constraint
  Model                                         Hosted     Reason
  ---------------------------------------------------------------------------
  Heartsync/NSFW-Uncensored-photo               0          Ethical constraint overrides
  BoobyBoobs/Flux_Lustly_AI_Uncensored_NSFW_V1  0          Ethical constraint overrides
  Landol4/nsfw-face-swap                        0          Ethical constraint overrides
  Menyu/wainsfw                                 0          Ethical constraint overrides
  Heartsync/Adult                               0          Ethical constraint overrides

  Hosted:  0/5
  Blocked: 5/5

  Unscreened derivative models with safety removed are blocked.
  The platform's functional goal is partially met.
  The general public's ethical goal takes precedence.


### Summary

| Scenario | Z3 Result | Meaning |
|---|---|---|
| All models screened | SAT | No conflict, both goals satisfied |
| Unscreened models with safety removed | UNSAT | Conflict: public ethical goal violated |
| Real data from ecosystem audit | UNSAT | Conflict confirmed with actual applications |
| Conflict resolution | Optimised | Ethical goal overrides functional goal |

This conflict is distinct from Notebooks 1 and 4:

- **Notebook 1** tested the design conflict — Stability AI making the safety checker removable
- **Notebook 4** tested the developer's internal conflict — generating content that violates their own ethical obligation
- **This notebook** tests the distribution conflict — a platform hosting models that harm a third party stakeholder who has no say in the process

The violated principle is utilitarian: the benefit of wide distribution is spread across many users, while the harm of hosting unscreened models 
with removed safety falls on specific identifiable individuals — women and minors — who receive none of the distribution benefit.

The general public's ethical goal takes precedence because they bear the full cost of the platform's design choice with no access to it and no 
mechanism for redress within it.